# Black Friday Sale Investigation
Assignment 5 · PySpark Data Engineering Challenge · Set 2

This notebook investigates sales performance, inventory movements and data quality using four imperfect source datasets.

## 1. Start Spark and define input paths
We start one Spark session and keep the input and output locations together.

In [1]:
# Start with the Spark pieces used throughout the notebook so the later transformations stay consistent.

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.window import Window
from pyspark.sql.functions import broadcast

# Creating one Spark session for all transformations and analyses.
spark = (
    SparkSession.builder
    .appName("BlackFridaySaleInvestigation")
    .master("local[*]")
    .getOrCreate()
)

# Reduce unnecessary Spark messages while keeping important warnings visible.
spark.sparkContext.setLogLevel("WARN")

# Keep all file locations in one place so paths are easy to update.
DATA_DIR = "data/raw"
OUTPUT_DIR = "output"

customers_path = f"{DATA_DIR}/customers.csv"
products_path = f"{DATA_DIR}/products.csv"
orders_path = f"{DATA_DIR}/orders.csv"
inventory_path = f"{DATA_DIR}/inventory_events.csv"

## 2. Define schemas and load source files
We declare the expected columns explicitly and initially read source values as strings to preserve imperfect data.

In [2]:
# I am keeping the expected source columns explicit here; it makes schema issues easier to spot early.

# Define the customer columns exactly as they appear in the source CSV.
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("region", StringType(), True),
    StructField("loyalty_tier", StringType(), True),
    StructField("signup_date", StringType(), True)
])

# Define the product columns without asking Spark to infer their data types.
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("list_price", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("supplier_id", StringType(), True)
])

# Keep order fields as strings for now so malformed numeric and timestamp values remain available.
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_time", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("discount_pct", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("channel", StringType(), True)
])

# Define the inventory event structure in the same way.
inventory_schema = StructType([
    StructField("inventory_event_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("warehouse", StringType(), True),
    StructField("stock_change", StringType(), True),
    StructField("event_type", StringType(), True)
])


# Reuse one function to load all CSV files with their declared schemas.
def ingest_csv(file_path, schema):
    return (
        spark.read
        .format("csv")
        .option("header", True)
        .option("mode", "PERMISSIVE")
        .schema(schema)
        .load(file_path)
    )


# Load each source file without modifying its original values.
customers_raw_df = ingest_csv(customers_path, customers_schema)
products_raw_df = ingest_csv(products_path, products_schema)
orders_raw_df = ingest_csv(orders_path, orders_schema)
inventory_raw_df = ingest_csv(inventory_path, inventory_schema)

print("All four source files loaded.")

All four source files loaded.


## 3. Parse important columns
We preserve the original text in raw columns and safely convert fields needed for calculations.

In [3]:
# The conversion helper is kept separate because the same type checks are needed across multiple datasets.

# This function keeps a copy of each source value before attempting its conversion.
def parse_columns(df, type_map):
    for column, spark_type in type_map.items():

        # Store the original text so we can investigate values that fail conversion.
        raw_column = f"{column}_raw"
        df = df.withColumn(raw_column, F.col(column))

        # try_cast converts invalid values to null instead of stopping the entire job.
        df = df.withColumn(
            column,
            F.expr(f"try_cast(`{raw_column}` AS {spark_type})")
        )

    return df


# Customer signup dates are required for the date consistency check.
customers_df = parse_columns(
    customers_raw_df,
    {"signup_date": "DATE"}
)

# Product list prices are parsed for product data-quality checks.
products_df = parse_columns(
    products_raw_df,
    {"list_price": "DOUBLE"}
)

# Parse order fields that are used in revenue and campaign analysis.
orders_df = parse_columns(
    orders_raw_df,
    {
        "order_time": "TIMESTAMP",
        "quantity": "INT",
        "unit_price": "DOUBLE",
        "discount_pct": "DOUBLE"
    }
)

# Parse inventory timestamps and stock changes for movement analysis.
inventory_df = parse_columns(
    inventory_raw_df,
    {
        "event_time": "TIMESTAMP",
        "stock_change": "INT"
    }
)

# Confirm that Spark assigned the intended data types.
orders_df.printSchema()
inventory_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_time: timestamp (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- order_status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- order_time_raw: string (nullable = true)
 |-- quantity_raw: string (nullable = true)
 |-- unit_price_raw: string (nullable = true)
 |-- discount_pct_raw: string (nullable = true)

root
 |-- inventory_event_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- warehouse: string (nullable = true)
 |-- stock_change: integer (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time_raw: string (nullable = true)
 |-- stock_change_raw: string (nullable = true)



In [4]:
# A quick schema check here helps catch incorrect types before doing any calculations.

customers_df.printSchema()
products_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- signup_date_raw: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- list_price: double (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- list_price_raw: string (nullable = true)



## 4. Check source size and record keys
We check the row counts, distinct keys and null keys before cleaning the data.

In [5]:
# These checks focus on the row grain and key columns, since duplicates can affect later joins.

# This reusable function checks the expected record grain of each dataset.
def show_dataset_profile(df, dataset_name, key_column):

    # Total rows tell us how many records were received from the source.
    total_rows = df.count()

    # Distinct keys help us identify repeated business identifiers.
    distinct_keys = df.select(key_column).distinct().count()

    # Null keys cannot reliably identify a customer, product, order or event.
    null_keys = df.filter(F.col(key_column).isNull()).count()

    print(
        f"{dataset_name}: rows={total_rows}, "
        f"distinct {key_column}={distinct_keys}, "
        f"null {key_column}={null_keys}"
    )


# Check the expected unique key in each source dataset.
show_dataset_profile(customers_df, "Customers", "customer_id")
show_dataset_profile(products_df, "Products", "product_id")
show_dataset_profile(orders_df, "Orders", "order_id")
show_dataset_profile(inventory_df, "Inventory", "inventory_event_id")

Customers: rows=1500, distinct customer_id=1500, null customer_id=0
Products: rows=100, distinct product_id=100, null product_id=0
Orders: rows=6003, distinct order_id=6000, null order_id=0
Inventory: rows=450, distinct inventory_event_id=450, null inventory_event_id=0


In [6]:
# This is a small visual check before moving into the more detailed quality analysis.

# Display a few records from each dataset to verify that ingestion and parsing look reasonable.
customers_df.show(5, truncate=False)
products_df.show(5, truncate=False)

+-----------+------+------------+-----------+---------------+
|customer_id|region|loyalty_tier|signup_date|signup_date_raw|
+-----------+------+------------+-----------+---------------+
|C00001     |North |Bronze      |2025-06-05 |2025-06-05     |
|C00002     |West  |silver      |2025-10-23 |2025-10-23     |
|C00003     |East  |Platinum    |2025-11-28 |2025-11-28     |
|C00004     |north |Bronze      |2024-10-09 |2024-10-09     |
|C00005     |North |silver      |2026-02-27 |2026-02-27     |
+-----------+------+------------+-----------+---------------+
only showing top 5 rows
+----------+--------------+-----------+----------+----------+-----------+--------------+
|product_id|product_name  |category   |list_price|brand     |supplier_id|list_price_raw|
+----------+--------------+-----------+----------+----------+-----------+--------------+
|PR0001    |Nova Tablet 1 |Electronics|536.17    |Northstar |SU004      |536.17        |
|PR0002    |Eco Lamp 2    |Toys       |326.68    |Northstar |S

In [7]:
# Keep this sample output small; the full dataset is better inspected through counts and summaries.

orders_df.show(5, truncate=False)
inventory_df.show(5, truncate=False)

+--------+-----------+----------+-------------------+--------+----------+------------+------------+-------+-------------------+------------+--------------+----------------+
|order_id|customer_id|product_id|order_time         |quantity|unit_price|discount_pct|order_status|channel|order_time_raw     |quantity_raw|unit_price_raw|discount_pct_raw|
+--------+-----------+----------+-------------------+--------+----------+------------+------------+-------+-------------------+------------+--------------+----------------+
|OR000001|C01132     |PR0080    |2025-11-27 21:15:00|1       |732.92    |0.2         |returned    |WEB    |2025-11-27 21:15:00|1           |732.92        |0.2             |
|OR000002|C00445     |PR0002    |2025-11-29 09:28:00|NULL    |483.51    |0.15        |returned    |mobile |2025-11-29 09:28:00|NULL        |483.51        |0.15            |
|OR000003|C01164     |PR0086    |2025-11-28 00:59:00|2       |742.24    |0.15        |completed   |mobile |2025-11-28 00:59:00|2       

## 5. Profile missing values and duplicate keys
We focus on fields that affect joins, revenue calculations and inventory analysis.

In [8]:
# Null counts give a first indication of where the source data may need extra handling.

# Count null values only in the columns that matter for this investigation.
def check_nulls(df, dataset_name, columns):
    print(f"\nNull counts - {dataset_name}")

    df.select([
        F.count(
            F.when(F.col(column).isNull(), 1)
        ).alias(column)
        for column in columns
    ]).show(truncate=False)


# Identify repeated non-null business keys without deleting any records.
def check_duplicate_keys(df, key_column, dataset_name):
    duplicates = (
        df.groupBy(key_column)
        .count()
        .filter(
            F.col(key_column).isNotNull()
            & (F.col("count") > 1)
        )
    )

    print(
        f"{dataset_name}: duplicate {key_column} values = "
        f"{duplicates.count()}"
    )

    return duplicates


# Check important customer and product fields.
check_nulls(
    customers_df,
    "Customers",
    ["customer_id", "region", "loyalty_tier", "signup_date"]
)

check_nulls(
    products_df,
    "Products",
    ["product_id", "category", "list_price"]
)

# Order fields are important because they directly affect sales calculations.
check_nulls(
    orders_df,
    "Orders",
    [
        "order_id", "customer_id", "product_id", "order_time",
        "quantity", "unit_price", "discount_pct",
        "order_status", "channel"
    ]
)

# Inventory fields determine whether an event can be analyzed reliably.
check_nulls(
    inventory_df,
    "Inventory",
    [
        "inventory_event_id", "product_id", "event_time",
        "warehouse", "stock_change", "event_type"
    ]
)


# Save duplicate-key summaries so they can be used during validation and cleaning.
duplicate_customer_keys = check_duplicate_keys(
    customers_df, "customer_id", "Customers"
)

duplicate_product_keys = check_duplicate_keys(
    products_df, "product_id", "Products"
)

duplicate_order_keys = check_duplicate_keys(
    orders_df, "order_id", "Orders"
)

duplicate_inventory_keys = check_duplicate_keys(
    inventory_df, "inventory_event_id", "Inventory"
)


Null counts - Customers
+-----------+------+------------+-----------+
|customer_id|region|loyalty_tier|signup_date|
+-----------+------+------------+-----------+
|0          |217   |228         |0          |
+-----------+------+------------+-----------+


Null counts - Products
+----------+--------+----------+
|product_id|category|list_price|
+----------+--------+----------+
|0         |0       |4         |
+----------+--------+----------+


Null counts - Orders
+--------+-----------+----------+----------+--------+----------+------------+------------+-------+
|order_id|customer_id|product_id|order_time|quantity|unit_price|discount_pct|order_status|channel|
+--------+-----------+----------+----------+--------+----------+------------+------------+-------+
|0       |0          |0         |1         |104     |92        |90          |0           |1237   |
+--------+-----------+----------+----------+--------+----------+------------+------------+-------+


Null counts - Inventory
+----------

In [9]:
# Range checks help distinguish values that are technically present from values that are usable.

# Check the numeric ranges before deciding which values should be treated as invalid.
orders_df.select(
    F.min("order_time").alias("first_order_time"),
    F.max("order_time").alias("last_order_time"),
    F.min("quantity").alias("minimum_quantity"),
    F.max("quantity").alias("maximum_quantity"),
    F.min("unit_price").alias("minimum_unit_price"),
    F.max("unit_price").alias("maximum_unit_price"),
    F.min("discount_pct").alias("minimum_discount"),
    F.max("discount_pct").alias("maximum_discount")
).show(truncate=False)


+-------------------+-------------------+----------------+----------------+------------------+------------------+----------------+----------------+
|first_order_time   |last_order_time    |minimum_quantity|maximum_quantity|minimum_unit_price|maximum_unit_price|minimum_discount|maximum_discount|
+-------------------+-------------------+----------------+----------------+------------------+------------------+----------------+----------------+
|2025-11-20 00:02:00|2025-11-30 23:58:00|-2              |10              |-15.0             |799.75            |0.0             |1.2             |
+-------------------+-------------------+----------------+----------------+------------------+------------------+----------------+----------------+



In [10]:
# Keep the source-level checks separate so the results can be reviewed before cleaning starts.



# Inspect order labels before standardizing their casing and spaces.
for column in ["order_status", "channel"]:
    print(f"\nOrders - {column}")
    orders_df.groupBy(column).count().orderBy(
        F.desc("count")
    ).show(truncate=False)




Orders - order_status
+------------+-----+
|order_status|count|
+------------+-----+
|completed   |1951 |
|Completed   |1064 |
|returned    |1032 |
|pending     |990  |
|cancelled   |966  |
+------------+-----+


Orders - channel
+-------+-----+
|channel|count|
+-------+-----+
|NULL   |1237 |
|store  |1219 |
|mobile |1208 |
|WEB    |1182 |
|web    |1157 |
+-------+-----+



In [11]:
# This check is useful as a baseline; later counts can be compared against it after filtering.


# Check product labels to identify inconsistent category or brand formatting.
for column in ["category", "brand"]:
    print(f"\nProducts - {column}")
    products_df.groupBy(column).count().orderBy(
        F.desc("count")
    ).show(truncate=False)




Products - category
+-----------+-----+
|category   |count|
+-----------+-----+
|Electronics|23   |
|Toys       |20   |
|Sports     |15   |
|Fashion    |14   |
|Beauty     |14   |
|Home       |13   |
|electronics|1    |
+-----------+-----+


Products - brand
+----------+-----+
|brand     |count|
+----------+-----+
|Northstar |24   |
|NovaTech  |21   |
|Everyday  |15   |
|UrbanLeaf |15   |
|Northstar |15   |
|NULL      |10   |
+----------+-----+



In [12]:
# Capture the result here rather than repeatedly scanning the source while reviewing the notebook.


# Check customer labels so missing and inconsistent groups remain visible.
for column in ["region", "loyalty_tier"]:
    print(f"\nCustomers - {column}")
    customers_df.groupBy(column).count().orderBy(
        F.desc("count")
    ).show(truncate=False)




Customers - region
+-------+-----+
|region |count|
+-------+-----+
|West   |240  |
|South  |234  |
|East   |224  |
|NULL   |217  |
|North  |211  |
|Central|191  |
|north  |183  |
+-------+-----+


Customers - loyalty_tier
+------------+-----+
|loyalty_tier|count|
+------------+-----+
|silver      |272  |
|Bronze      |271  |
|Platinum    |266  |
|Gold        |235  |
|NULL        |228  |
|Silver      |228  |
+------------+-----+



In [13]:
# These checks help establish whether the lookup keys are safe to use in joins.


# Inspect the event types and warehouses recorded in the inventory file.
for column in ["warehouse", "event_type"]:
    print(f"\nInventory - {column}")
    inventory_df.groupBy(column).count().orderBy(
        F.desc("count")
    ).show(truncate=False)


Inventory - warehouse
+---------+-----+
|warehouse|count|
+---------+-----+
|WH-S     |127  |
|WH-E     |112  |
|WH-N     |109  |
|WH-W     |102  |
+---------+-----+


Inventory - event_type
+---------------+-----+
|event_type     |count|
+---------------+-----+
|sale_adjustment|102  |
|damage         |98   |
|restock        |91   |
|RESTOCK        |87   |
|return         |72   |
+---------------+-----+



## 6. Check reference integrity and signup dates
We identify orders and inventory events that refer to missing master records.

In [14]:
# Build the reference checks before joining so the quality review does not accidentally multiply rows.

# Build distinct reference lists so duplicate master records do not multiply join results.
customer_keys = (
    customers_df
    .filter(F.col("customer_id").isNotNull())
    .select("customer_id")
    .distinct()
)

product_keys = (
    products_df
    .filter(F.col("product_id").isNotNull())
    .select("product_id")
    .distinct()
)

# A left-anti join returns source rows whose reference key does not exist in the master list.
unknown_order_customers_df = orders_df.join(
    customer_keys,
    "customer_id",
    "left_anti"
)

unknown_order_products_df = orders_df.join(
    product_keys,
    "product_id",
    "left_anti"
)

unknown_inventory_products_df = inventory_df.join(
    product_keys,
    "product_id",
    "left_anti"
)

# Report unmatched references without removing them from the source data.
print(
    "Orders with unknown customer IDs:",
    unknown_order_customers_df.count()
)

print(
    "Orders with unknown product IDs:",
    unknown_order_products_df.count()
)

print(
    "Inventory events with unknown product IDs:",
    unknown_inventory_products_df.count()
)

Orders with unknown customer IDs: 0
Orders with unknown product IDs: 1
Inventory events with unknown product IDs: 1


In [15]:
# Looking at the first order date per customer gives a simple view of customer activity history.

# Find the earliest recorded order date for each customer.
first_customer_orders_df = (
    orders_df
    .filter(F.col("customer_id").isNotNull())
    .groupBy("customer_id")
    .agg(
        F.min("order_time").alias("first_order_time")
    )
)

# Join the first order date to customer signup information for comparison.
signup_order_check_df = customers_df.join(
    first_customer_orders_df,
    "customer_id",
    "left"
)

# A first order before signup may indicate a source-date inconsistency.
customers_ordering_before_signup_df = signup_order_check_df.filter(
    F.col("signup_date").isNotNull()
    & F.col("first_order_time").isNotNull()
    & (
        F.col("signup_date")
        > F.to_date("first_order_time")
    )
)

print(
    "Customers whose signup date is after their first recorded order:",
    customers_ordering_before_signup_df.count()
)

customers_ordering_before_signup_df.show(10, truncate=False)

Customers whose signup date is after their first recorded order: 244
+-----------+-------+------------+-----------+---------------+-------------------+
|customer_id|region |loyalty_tier|signup_date|signup_date_raw|first_order_time   |
+-----------+-------+------------+-----------+---------------+-------------------+
|C00855     |South  |Gold        |2026-01-02 |2026-01-02     |2025-11-21 12:46:00|
|C00310     |West   |Platinum    |2025-12-28 |2025-12-28     |2025-11-21 05:25:00|
|C00195     |north  |Platinum    |2026-04-26 |2026-04-26     |2025-11-21 09:36:00|
|C00261     |West   |Gold        |2026-05-17 |2026-05-17     |2025-11-20 20:22:00|
|C00572     |NULL   |silver      |2026-07-21 |2026-07-21     |2025-11-22 04:43:00|
|C00778     |North  |NULL        |2026-03-22 |2026-03-22     |2025-11-28 09:32:00|
|C00244     |South  |NULL        |2026-03-11 |2026-03-11     |2025-11-20 04:41:00|
|C00951     |NULL   |NULL        |2026-08-16 |2026-08-16     |2025-11-20 01:31:00|
|C00311     |North

## 7. Standardize text labels
We normalize casing and spaces in grouping columns while retaining their original values.

In [16]:
# Preserve the incoming labels while standardizing them, so the original values are still available for review.

# Apply the selected text-cleaning rule and preserve the original label in a raw column.
def standardize_text_columns(df, rules):

    for column, rule in rules.items():

        # Keep the source label before changing its format.
        raw_column = f"{column}_raw"

        if raw_column not in df.columns:
            df = df.withColumn(raw_column, F.col(column))

        # Lowercase is useful for labels such as channel and order status.
        if rule == "lower":
            df = df.withColumn(
                column,
                F.lower(F.trim(F.col(column)))
            )

        # Uppercase is used for inventory event and warehouse labels.
        elif rule == "upper":
            df = df.withColumn(
                column,
                F.upper(F.trim(F.col(column)))
            )

        # Title case makes category, region and loyalty labels consistent.
        elif rule == "title":
            df = df.withColumn(
                column,
                F.initcap(F.trim(F.col(column)))
            )

        # Trim removes unwanted spaces without changing the label's case.
        elif rule == "trim":
            df = df.withColumn(
                column,
                F.trim(F.col(column))
            )

        else:
            raise ValueError(f"Unsupported text rule: {rule}")

    return df


# Standardize customer labels but do not replace missing regions or loyalty tiers.
customers_clean_df = standardize_text_columns(
    customers_df,
    {
        "region": "title",
        "loyalty_tier": "title"
    }
)

# Standardize product categories so values like electronics and Electronics match.
products_clean_df = standardize_text_columns(
    products_df,
    {
        "category": "title",
        "brand": "trim"
    }
)

# Normalize order status and sales channel for consistent grouping.
orders_clean_df = standardize_text_columns(
    orders_df,
    {
        "order_status": "lower",
        "channel": "lower"
    }
)

# Keep inventory event labels and warehouse names in a consistent uppercase format.
inventory_clean_df = standardize_text_columns(
    inventory_df,
    {
        "event_type": "upper",
        "warehouse": "upper"
    }
)

In [17]:
# These grouped counts make it easier to see whether spelling and casing variations were consolidated.

# Display standardized labels to confirm that casing differences have been combined.
print("Standardized product categories:")
products_clean_df.groupBy("category").count().show()

print("Standardized inventory event types:")
inventory_clean_df.groupBy("event_type").count().show()



Standardized product categories:
+-----------+-----+
|   category|count|
+-----------+-----+
|       Home|   13|
|    Fashion|   14|
|     Sports|   15|
|Electronics|   24|
|     Beauty|   14|
|       Toys|   20|
+-----------+-----+

Standardized inventory event types:
+---------------+-----+
|     event_type|count|
+---------------+-----+
|         DAMAGE|   98|
|         RETURN|   72|
|        RESTOCK|  178|
|SALE_ADJUSTMENT|  102|
+---------------+-----+



In [18]:
# Check the other text dimensions as well; inconsistent labels can split what should be one group.

print("Standardized customer regions:")
customers_clean_df.groupBy("region").count().show()

print("Standardized sales channels:")
orders_clean_df.groupBy("channel").count().show()

Standardized customer regions:
+-------+-----+
| region|count|
+-------+-----+
|   NULL|  217|
|  South|  234|
|Central|  191|
|   East|  224|
|   West|  240|
|  North|  394|
+-------+-----+

Standardized sales channels:
+-------+-----+
|channel|count|
+-------+-----+
|   NULL| 1237|
| mobile| 1208|
|  store| 1219|
|    web| 2339|
+-------+-----+



## 8. Identify exact duplicate order records
We remove only extra copies of completely identical rows and keep them separately for audit.

In [19]:
# Exact duplicates are handled first, before checking whether an ID is reused across different records.

# Keep one copy of each completely identical order row.
orders_without_exact_dupes_df = orders_clean_df.dropDuplicates()

# exceptAll returns the extra copies removed by dropDuplicates.
exact_duplicate_order_rows_df = orders_clean_df.exceptAll(
    orders_without_exact_dupes_df
)

print(
    "Extra exact duplicate order rows:",
    exact_duplicate_order_rows_df.count()
)

# Check whether an order ID is still repeated after exact duplicates are removed.
remaining_duplicate_order_ids_df = (
    orders_without_exact_dupes_df
    .groupBy("order_id")
    .count()
    .filter(
        F.col("order_id").isNotNull()
        & (F.col("count") > 1)
    )
)

# Keep all rows belonging to a repeated order ID for the quarantine step.
duplicate_order_id_rows_df = orders_without_exact_dupes_df.join(
    remaining_duplicate_order_ids_df.select("order_id"),
    "order_id",
    "inner"
)

print(
    "Rows with repeated order IDs after exact deduplication:",
    duplicate_order_id_rows_df.count()
)

Extra exact duplicate order rows: 3
Rows with repeated order IDs after exact deduplication: 0


## 9. Validate orders and create a quarantine dataset
We flag records that cannot safely contribute to revenue and retain a reason for each rejected row.

In [20]:
# Keep the parsed discount alongside the source value so invalid entries are not silently lost.

# Keep the parsed discount separately so the original numeric value remains available for review.
orders_clean_df = orders_without_exact_dupes_df.withColumn(
    "discount_pct_parsed",
    F.col("discount_pct")
)

# Only discounts from 0 to 1 are accepted; 0.20 represents a 20% discount.
orders_clean_df = orders_clean_df.withColumn(
    "discount_pct",
    F.when(
        F.col("discount_pct_parsed").between(0.0, 1.0),
        F.col("discount_pct_parsed")
    )
)


In [21]:
# The flags created here are used to explain why an order is accepted or sent to quarantine.


# These are the only order statuses used in this investigation.
known_statuses = ["completed", "returned", "cancelled", "pending"]

# Prepare distinct keys so the quality joins cannot multiply order records.
duplicate_order_id_keys = remaining_duplicate_order_ids_df.select(
    "order_id"
).distinct()

unknown_customer_order_ids = (
    unknown_order_customers_df
    .filter(F.col("order_id").isNotNull())
    .select("order_id")
    .distinct()
    .withColumn("unknown_customer", F.lit(True))
)

unknown_product_order_ids = (
    unknown_order_products_df
    .filter(F.col("order_id").isNotNull())
    .select("order_id")
    .distinct()
    .withColumn("unknown_product", F.lit(True))
)

# Attach the three reference flags to each order using left joins.
orders_with_quality_df = (
    orders_clean_df
    .join(
        duplicate_order_id_keys.withColumn(
            "has_duplicate_order_id", F.lit(True)
        ),
        "order_id",
        "left"
    )
    .join(unknown_customer_order_ids, "order_id", "left")
    .join(unknown_product_order_ids, "order_id", "left")
)


In [22]:
# This cell brings the order validation steps together before the final valid and rejected datasets are created.

# Keep the parsed discount separately so the original numeric value remains available for review.
orders_clean_df = orders_without_exact_dupes_df.withColumn(
    "discount_pct_parsed",
    F.col("discount_pct")
)

# Only discounts from 0 to 1 are accepted; 0.20 represents a 20% discount.
orders_clean_df = orders_clean_df.withColumn(
    "discount_pct",
    F.when(
        F.col("discount_pct_parsed").between(0.0, 1.0),
        F.col("discount_pct_parsed")
    )
)

# These are the only order statuses used in this investigation.
known_statuses = ["completed", "returned", "cancelled", "pending"]

# Prepare distinct keys so the quality joins cannot multiply order records.
duplicate_order_id_keys = remaining_duplicate_order_ids_df.select(
    "order_id"
).distinct()

unknown_customer_order_ids = (
    unknown_order_customers_df
    .filter(F.col("order_id").isNotNull())
    .select("order_id")
    .distinct()
    .withColumn("unknown_customer", F.lit(True))
)

unknown_product_order_ids = (
    unknown_order_products_df
    .filter(F.col("order_id").isNotNull())
    .select("order_id")
    .distinct()
    .withColumn("unknown_product", F.lit(True))
)

# Attach the three reference flags to each order using left joins.
orders_with_quality_df = (
    orders_clean_df
    .join(
        duplicate_order_id_keys.withColumn(
            "has_duplicate_order_id", F.lit(True)
        ),
        "order_id",
        "left"
    )
    .join(unknown_customer_order_ids, "order_id", "left")
    .join(unknown_product_order_ids, "order_id", "left")
)

# Build one readable reason column containing every issue found for a row.
orders_with_quality_df = orders_with_quality_df.withColumn(
    "quality_issue",
    F.concat_ws(
        ", ",
        F.when(F.col("order_id").isNull(), "Missing order_id"),
        F.when(
            F.col("has_duplicate_order_id").isNotNull(),
            "Repeated order_id"
        ),
        F.when(F.col("customer_id").isNull(), "Missing customer_id"),
        F.when(
            F.col("unknown_customer").isNotNull(),
            "Unknown customer_id"
        ),
        F.when(F.col("product_id").isNull(), "Missing product_id"),
        F.when(
            F.col("unknown_product").isNotNull(),
            "Unknown product_id"
        ),
        F.when(
            F.col("order_time").isNull(),
            "Missing or malformed order_time"
        ),
        F.when(
            F.col("quantity").isNull() | (F.col("quantity") <= 0),
            "Invalid quantity"
        ),
        F.when(
            F.col("unit_price").isNull() | (F.col("unit_price") <= 0),
            "Invalid unit_price"
        ),
        F.when(
            F.col("discount_pct").isNull(),
            "Missing or invalid discount"
        ),
        F.when(
            F.col("order_status").isNull()
            | (~F.col("order_status").isin(*known_statuses)),
            "Missing or unknown order_status"
        )
    )
).drop(
    "has_duplicate_order_id",
    "unknown_customer",
    "unknown_product"
)

# Keep rejected records separately so the reasons can be reviewed and audited.
quarantine_orders_df = orders_with_quality_df.filter(
    F.col("quality_issue") != ""
)

# Only records that pass all required checks can contribute to sales metrics.
valid_orders_df = orders_with_quality_df.filter(
    F.col("quality_issue") == ""
)

print("Rows after exact duplicate handling:", orders_clean_df.count())
print("Valid orders:", valid_orders_df.count())
print("Quarantined orders:", quarantine_orders_df.count())

Rows after exact duplicate handling: 6000
Valid orders: 5025
Quarantined orders: 975


In [23]:
# Review a sample of rejected rows with the reason column, rather than dropping these records without an audit trail.

# Display rejected orders with their original values and all detected quality reasons.
quarantine_orders_df.select(
    "order_id",
    "customer_id",
    "product_id",
    "quantity_raw",
    "unit_price_raw",
    "discount_pct_raw",
    "discount_pct_parsed",
    "quality_issue"
).show(30, truncate=False)

+--------+-----------+----------+------------+--------------+----------------+-------------------+---------------------------+
|order_id|customer_id|product_id|quantity_raw|unit_price_raw|discount_pct_raw|discount_pct_parsed|quality_issue              |
+--------+-----------+----------+------------+--------------+----------------+-------------------+---------------------------+
|OR000423|C01111     |PR0090    |1           |473.17        |1.2             |1.2                |Missing or invalid discount|
|OR000630|C00341     |PR0031    |1           |118.92        |1.2             |1.2                |Missing or invalid discount|
|OR003311|C00188     |PR0076    |1           |220.58        |1.2             |1.2                |Missing or invalid discount|
|OR001306|C00812     |PR0033    |4           |79.93         |1.2             |1.2                |Missing or invalid discount|
|OR002678|C00109     |PR0028    |3           |585.54        |1.2             |1.2                |Missing or in

## 10. Calculate realized sales and revenue
We calculate revenue only from valid completed orders using the actual order price and validated discount.

In [24]:
# Revenue is calculated only after the order quality rules have been applied.

# Completed orders represent realized sales under the selected business rule.
sales_df = (
    valid_orders_df
    .filter(F.col("order_status") == "completed")
    .withColumn(
        "revenue",
        F.round(
            F.col("quantity")
            * F.col("unit_price")
            * (1 - F.col("discount_pct")),
            2
        )
    )

    # Extract date and hour once so later campaign analysis can reuse them.
    .withColumn("order_date", F.to_date("order_time"))
    .withColumn("order_hour", F.hour("order_time"))
)


In [25]:
# Keeping non-completed statuses separate avoids mixing returns, cancellations and pending orders into sales.


# Keep other valid order statuses separate instead of treating them as sales.
returned_orders_df = valid_orders_df.filter(
    F.col("order_status") == "returned"
)

cancelled_orders_df = valid_orders_df.filter(
    F.col("order_status") == "cancelled"
)

pending_orders_df = valid_orders_df.filter(
    F.col("order_status") == "pending"
)

print("Completed sales rows:", sales_df.count())
print("Returned orders:", returned_orders_df.count())
print("Cancelled orders:", cancelled_orders_df.count())
print("Pending orders:", pending_orders_df.count())

Completed sales rows: 2537
Returned orders: 853
Cancelled orders: 801
Pending orders: 834


In [26]:
# This comparison shows why applying the business rules matters, not just how the arithmetic is performed.

# Calculate a simple baseline using all source orders without applying discounts or quality rules.
naive_revenue_df = orders_df.agg(
    F.round(
        F.sum(
            F.col("quantity") * F.col("unit_price")
        ),
        2
    ).alias("naive_revenue")
)

# Calculate the final revenue using only validated completed orders.
defensible_revenue_df = sales_df.agg(
    F.round(F.sum("revenue"), 2).alias("defensible_revenue")
)

# Put both totals side by side to show how business rules affect the result.
revenue_comparison_df = (
    naive_revenue_df
    .crossJoin(defensible_revenue_df)
    .withColumn(
        "difference",
        F.round(
            F.col("naive_revenue") - F.col("defensible_revenue"),
            2
        )
    )
)

revenue_comparison_df.show(truncate=False)

+-------------+------------------+----------+
|naive_revenue|defensible_revenue|difference|
+-------------+------------------+----------+
|8103007.97   |2946086.89        |5156921.08|
+-------------+------------------+----------+



## 11. Prepare safe dimension tables and enrich sales
We remove duplicate keys from lookup tables before joining and verify that sales rows are not multiplied.

In [27]:
# The lookup tables are prepared independently so the sales fact rows remain at their original grain.

# Keep all records belonging to duplicate product keys for quality review.
duplicate_product_rows_df = products_clean_df.join(
    duplicate_product_keys.select("product_id").distinct(),
    "product_id",
    "inner"
)

# Keep all records belonging to duplicate customer keys for quality review.
duplicate_customer_rows_df = customers_clean_df.join(
    duplicate_customer_keys.select("customer_id").distinct(),
    "customer_id",
    "inner"
)

# Exclude duplicate product keys from the lookup side to prevent row multiplication.
products_unique_df = products_clean_df.join(
    duplicate_product_keys.select("product_id").distinct(),
    "product_id",
    "left_anti"
)

# Apply the same protection to customer lookup records.
customers_unique_df = customers_clean_df.join(
    duplicate_customer_keys.select("customer_id").distinct(),
    "customer_id",
    "left_anti"
)

# Add product and customer details while keeping every valid sales row.
sales_enriched_df = (
    sales_df
    .join(
        products_unique_df.select(
            "product_id", "product_name", "category", "brand"
        ),
        "product_id",
        "left"
    )
    .join(
        customers_unique_df.select(
            "customer_id", "region", "loyalty_tier"
        ),
        "customer_id",
        "left"
    )
)

# A many-to-one lookup should not change the number of sales records.
sales_rows_before_join = sales_df.count()
sales_rows_after_join = sales_enriched_df.count()

print("Sales rows before joins:", sales_rows_before_join)
print("Sales rows after joins:", sales_rows_after_join)


Sales rows before joins: 2537
Sales rows after joins: 2537


In [28]:
# Check the join result before trusting any product or customer breakdowns built from it.


assert sales_rows_before_join == sales_rows_after_join, (
    "A dimension join changed the sales row count. Check dimension keys."
)

# Missing product or customer details indicate unmatched or ambiguous lookup records.
unmatched_product_sales_df = sales_enriched_df.filter(
    F.col("product_name").isNull()
)

unmatched_customer_sales_df = sales_enriched_df.filter(
    F.col("loyalty_tier").isNull()
)

print(
    "Sales rows with unavailable product details:",
    unmatched_product_sales_df.count()
)

print(
    "Sales rows with unavailable customer details:",
    unmatched_customer_sales_df.count()
)

Sales rows with unavailable product details: 0
Sales rows with unavailable customer details: 394


## 12. Analyze product and category performance
We compare realized revenue, order counts, units sold and average discounts across products and categories.

In [29]:
# Use realized revenue as the main measure, while keeping order count and units sold for context.

# Summarize category performance using the enriched sales records.
category_performance_df = (
    sales_enriched_df
    .groupBy("category")
    .agg(
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("units_sold"),
        F.round(
            F.sum("revenue") / F.countDistinct("order_id"),
            2
        ).alias("average_order_value"),
        F.round(
            F.avg("discount_pct") * 100,
            2
        ).alias("average_discount_pct")
    )
    .orderBy(F.desc("total_revenue"))
)


In [30]:
# Product-level aggregation gives a more detailed view than the category summary above.


# Summarize each product so we can identify revenue and volume leaders.
product_performance_df = (
    sales_enriched_df
    .groupBy("product_id", "product_name", "category")
    .agg(
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("units_sold"),
        F.round(
            F.avg("discount_pct") * 100,
            2
        ).alias("average_discount_pct")
    )
    .orderBy(F.desc("total_revenue"))
)

In [31]:
# Display both levels of performance so the category results can be compared with individual products.

# Display category-level results first to compare the larger business groups.
category_performance_df.show(truncate=False)

# Display the leading products without printing the entire product list.
product_performance_df.show(20, truncate=False)

+-----------+-------------+-----------+----------+-------------------+--------------------+
|category   |total_revenue|order_count|units_sold|average_order_value|average_discount_pct|
+-----------+-------------+-----------+----------+-------------------+--------------------+
|Electronics|708814.73    |597        |2110      |1187.29            |18.79               |
|Toys       |555280.62    |502        |1717      |1106.14            |19.19               |
|Sports     |519506.21    |425        |1486      |1222.37            |19.01               |
|Fashion    |419770.68    |328        |1212      |1279.79            |18.57               |
|Beauty     |400890.63    |376        |1265      |1066.2             |18.67               |
|Home       |341824.02    |309        |1041      |1106.23            |19.06               |
+-----------+-------------+-----------+----------+-------------------+--------------------+

+----------+-------------------+-----------+-------------+-----------+---------

## 13. Compare channel, region and loyalty performance
We reuse one aggregation function to compare sales across different customer and sales dimensions.

In [32]:
# Reusing one function keeps the measures consistent across the different business dimensions.

# Reuse the same measures so channel, region and loyalty results are comparable.
def performance_by_dimension(df, dimension):
    return (
        df.groupBy(dimension)
        .agg(
            F.round(F.sum("revenue"), 2).alias("total_revenue"),
            F.countDistinct("order_id").alias("order_count"),
            F.sum("quantity").alias("units_sold"),
            F.round(
                F.sum("revenue") / F.countDistinct("order_id"),
                2
            ).alias("average_order_value")
        )
        .orderBy(F.desc("total_revenue"))
    )


In [33]:
# Run the same calculation for each dimension instead of maintaining separate aggregation logic.



# Compare sales channels after standardizing their text labels.
channel_performance_df = performance_by_dimension(
    sales_enriched_df, "channel"
)

# Compare customer regions while keeping missing regions visible.
region_performance_df = performance_by_dimension(
    sales_enriched_df, "region"
)

# Compare revenue contribution across loyalty tiers.
loyalty_performance_df = performance_by_dimension(
    sales_enriched_df, "loyalty_tier"
)

In [34]:
# These outputs are intended for comparison, so the same revenue definition is used in each one.

# Display each dimension separately so its output is easy to inspect.
print("Performance by channel:")
channel_performance_df.show(truncate=False)

print("Performance by customer region:")
region_performance_df.show(truncate=False)

print("Performance by loyalty tier:")
loyalty_performance_df.show(truncate=False)

Performance by channel:
+-------+-------------+-----------+----------+-------------------+
|channel|total_revenue|order_count|units_sold|average_order_value|
+-------+-------------+-----------+----------+-------------------+
|web    |1131708.17   |962        |3366      |1176.41            |
|mobile |620718.75    |531        |1890      |1168.96            |
|store  |618534.41    |534        |1881      |1158.3             |
|NULL   |575125.56    |510        |1694      |1127.7             |
+-------+-------------+-----------+----------+-------------------+

Performance by customer region:
+-------+-------------+-----------+----------+-------------------+
|region |total_revenue|order_count|units_sold|average_order_value|
+-------+-------------+-----------+----------+-------------------+
|North  |784228.11    |641        |2371      |1223.44            |
|West   |480678.3     |401        |1329      |1198.7             |
|East   |448277.06    |396        |1396      |1132.01            |
|NULL

## 14. Investigate unusual discounts and transactions
We separate invalid values from high-value transactions that only need business review.

In [35]:
# These are review flags rather than automatic rejection rules; a high discount can still be legitimate.

# A discount of 50% or more is unusual enough to review, but it is not automatically invalid.
high_discount_orders_df = orders_clean_df.filter(
    F.col("discount_pct_parsed") >= 0.50
)

# Build reasons for unusual transactions without changing their original values.
suspicious_orders_df = (
    orders_clean_df
    .withColumn(
        "anomaly_reason",
        F.concat_ws(
            ", ",
            F.when(
                F.col("quantity") < 0,
                "Negative quantity"
            ),
            F.when(
                F.col("unit_price") < 0,
                "Negative unit price"
            ),
            F.when(
                F.col("discount_pct_parsed") >= 0.50,
                "High discount - review"
            ),
            F.when(
                F.col("quantity") > 100,
                "Extreme quantity - review"
            ),
            F.when(
                F.col("discount_pct_parsed") > 1,
                "Discount above 100% - invalid"
            )
        )
    )
    .filter(F.col("anomaly_reason") != "")
)

print(
    "Orders with discounts of 50% or more:",
    high_discount_orders_df.count()
)

# Show source values so unusual transactions can be traced back to the input.
suspicious_orders_df.select(
    "order_id",
    "product_id",
    "quantity_raw",
    "unit_price_raw",
    "discount_pct_raw",
    "anomaly_reason"
).show(50, truncate=False)

Orders with discounts of 50% or more: 1451
+--------+----------+------------+--------------+----------------+-----------------------------------------------------+
|order_id|product_id|quantity_raw|unit_price_raw|discount_pct_raw|anomaly_reason                                       |
+--------+----------+------------+--------------+----------------+-----------------------------------------------------+
|OR000423|PR0090    |1           |473.17        |1.2             |High discount - review, Discount above 100% - invalid|
|OR000630|PR0031    |1           |118.92        |1.2             |High discount - review, Discount above 100% - invalid|
|OR000940|PR0005    |10          |516.81        |0.5             |High discount - review                               |
|OR001341|PR0056    |2           |637.49        |0.5             |High discount - review                               |
|OR003300|PR0087    |3           |352.69        |0.5             |High discount - review                      

## 15. Validate inventory events and create a quarantine
We retain events with missing fields, duplicate IDs or unknown product references for review.

In [36]:
# Inventory validation follows the same audit approach as orders: flag issues and retain the affected rows.

# Prepare distinct duplicate event IDs so the quality join remains one-to-one.
duplicate_inventory_id_keys = (
    duplicate_inventory_keys
    .select("inventory_event_id")
    .distinct()
    .withColumn("has_duplicate_event_id", F.lit(True))
)

# Prepare event IDs whose product references are not found in the product master.
unknown_inventory_event_ids = (
    unknown_inventory_products_df
    .filter(F.col("inventory_event_id").isNotNull())
    .select("inventory_event_id")
    .distinct()
    .withColumn("has_unknown_product", F.lit(True))
)

# Attach quality flags to each inventory record without collecting IDs to the driver.
inventory_with_quality_df = (
    inventory_clean_df
    .join(
        duplicate_inventory_id_keys,
        "inventory_event_id",
        "left"
    )
    .join(
        unknown_inventory_event_ids,
        "inventory_event_id",
        "left"
    )
)

# Record all issues found in each event so one row can have multiple reasons.
inventory_with_quality_df = inventory_with_quality_df.withColumn(
    "quality_issue",
    F.concat_ws(
        ", ",
        F.when(
            F.col("inventory_event_id").isNull(),
            "Missing inventory_event_id"
        ),
        F.when(
            F.col("has_duplicate_event_id").isNotNull(),
            "Duplicate inventory_event_id"
        ),
        F.when(
            F.col("product_id").isNull(),
            "Missing product_id"
        ),
        F.when(
            F.col("has_unknown_product").isNotNull(),
            "Unknown product_id"
        ),
        F.when(
            F.col("event_time").isNull(),
            "Missing or malformed event_time"
        ),
        F.when(
            F.col("stock_change").isNull(),
            "Missing or malformed stock_change"
        ),
        F.when(
            F.col("event_type").isNull(),
            "Missing event_type"
        ),
        F.when(
            F.col("warehouse").isNull(),
            "Missing warehouse"
        )
    )
).drop(
    "has_duplicate_event_id",
    "has_unknown_product"
)

# Keep every event with at least one issue in the quarantine dataset.
inventory_quarantine_df = inventory_with_quality_df.filter(
    F.col("quality_issue") != ""
)

# Only fully valid events are used for inventory movement summaries.
valid_inventory_df = inventory_with_quality_df.filter(
    F.col("quality_issue") == ""
)

print("Valid inventory events:", valid_inventory_df.count())
print("Quarantined inventory events:", inventory_quarantine_df.count())

Valid inventory events: 439
Quarantined inventory events: 11


In [37]:
# A sample of quarantined inventory events helps confirm that the recorded reasons are understandable.

# Inspect the reasons for rejected inventory events before analyzing stock movement.
inventory_quarantine_df.select(
    "inventory_event_id",
    "product_id",
    "event_time",
    "warehouse",
    "stock_change",
    "event_type",
    "quality_issue"
).show(11, truncate=False)

+------------------+----------+-------------------+---------+------------+---------------+---------------------------------+
|inventory_event_id|product_id|event_time         |warehouse|stock_change|event_type     |quality_issue                    |
+------------------+----------+-------------------+---------+------------+---------------+---------------------------------+
|IV00010           |PR8888    |2025-11-30 15:00:00|WH-W     |-5          |RETURN         |Unknown product_id               |
|IV00019           |PR0032    |NULL               |WH-W     |20          |RESTOCK        |Missing or malformed event_time  |
|IV00043           |PR0091    |2025-11-21 00:00:00|WH-W     |NULL        |SALE_ADJUSTMENT|Missing or malformed stock_change|
|IV00106           |PR0030    |2025-11-26 16:00:00|WH-S     |NULL        |RETURN         |Missing or malformed stock_change|
|IV00222           |PR0015    |2025-11-27 09:00:00|WH-W     |NULL        |RESTOCK        |Missing or malformed stock_change|


## 16. Summarize inventory movements
We analyze stock changes by product, warehouse, event type and date.

In [38]:
# Summarize only the valid events so malformed or ambiguous records do not affect movement totals.

# Summarize the overall movement recorded for each product, warehouse and event type.
inventory_summary_df = (
    valid_inventory_df
    .groupBy("product_id", "warehouse", "event_type")
    .agg(
        F.sum("stock_change").alias("net_stock_change"),
        F.count("*").alias("event_count")
    )
    .orderBy("product_id", "warehouse", "event_type")
)

# Extract the event date so movement can also be studied over time.
inventory_with_date_df = valid_inventory_df.withColumn(
    "event_date",
    F.to_date("event_time")
)


In [39]:
# The daily view keeps warehouse and event type in the grouping, which helps avoid hiding operational differences.


# Preserve all requested dimensions in the daily inventory summary.
daily_inventory_df = (
    inventory_with_date_df
    .groupBy(
        "event_date",
        "product_id",
        "warehouse",
        "event_type"
    )
    .agg(
        F.sum("stock_change").alias("daily_stock_change"),
        F.count("*").alias("event_count")
    )
    .orderBy("event_date", "product_id", "warehouse")
)

In [40]:
# These displays provide a quick check of both the overall and date-level inventory results.

# Display the overall stock movement summary.
inventory_summary_df.show(20, truncate=False)

# Display daily stock movement to identify when and where changes occurred.
daily_inventory_df.show(20, truncate=False)

+----------+---------+---------------+----------------+-----------+
|product_id|warehouse|event_type     |net_stock_change|event_count|
+----------+---------+---------------+----------------+-----------+
|PR0001    |WH-E     |RESTOCK        |-50             |1          |
|PR0001    |WH-N     |RESTOCK        |10              |1          |
|PR0001    |WH-S     |RESTOCK        |-10             |1          |
|PR0001    |WH-S     |SALE_ADJUSTMENT|50              |1          |
|PR0001    |WH-W     |RESTOCK        |30              |2          |
|PR0002    |WH-E     |RESTOCK        |40              |3          |
|PR0002    |WH-N     |DAMAGE         |-10             |1          |
|PR0002    |WH-W     |RESTOCK        |0               |2          |
|PR0003    |WH-S     |RETURN         |-5              |1          |
|PR0003    |WH-S     |SALE_ADJUSTMENT|-10             |1          |
|PR0003    |WH-W     |RETURN         |-50             |1          |
|PR0004    |WH-N     |RESTOCK        |-30       

## 17. Compare sales with recorded inventory movement
We compare daily units sold and stock movement by product as operational context, not as a stock reconciliation.

In [41]:
# The comparison is aligned by product and date, but it should be read as context rather than a stock balance check.

# Calculate daily sales quantity and revenue for each product.
daily_product_sales_df = (
    sales_df
    .groupBy("order_date", "product_id")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("revenue"), 2).alias("daily_revenue")
    )
)


# Calculate the net inventory movement recorded for each product and event date.
daily_product_stock_df = (
    inventory_with_date_df
    .groupBy("event_date", "product_id")
    .agg(
        F.sum("stock_change").alias("net_stock_change")
    )
)

# Match sales and stock movements using the same product and calendar date.
sales_inventory_comparison_df = (
    daily_product_sales_df.alias("s")
    .join(
        daily_product_stock_df.alias("i"),
        (F.col("s.order_date") == F.col("i.event_date"))
        & (F.col("s.product_id") == F.col("i.product_id")),
        "left"
    )
    .select(
        F.col("s.order_date").alias("order_date"),
        F.col("s.product_id").alias("product_id"),
        "units_sold",
        "daily_revenue",
        "net_stock_change"
    )
)

sales_inventory_comparison_df.show(20, truncate=False)

+----------+----------+----------+-------------+----------------+
|order_date|product_id|units_sold|daily_revenue|net_stock_change|
+----------+----------+----------+-------------+----------------+
|2025-11-23|PR0074    |37        |15964.55     |NULL            |
|2025-11-26|PR0094    |16        |1750.37      |-30             |
|2025-11-20|PR0076    |1         |547.13       |NULL            |
|2025-11-23|PR0080    |10        |4633.23      |NULL            |
|2025-11-26|PR0055    |11        |5010.39      |-5              |
|2025-11-26|PR0032    |20        |3513.87      |20              |
|2025-11-27|PR0015    |29        |13050.89     |-10             |
|2025-11-23|PR0088    |7         |3101.0       |NULL            |
|2025-11-20|PR0074    |19        |5570.75      |NULL            |
|2025-11-27|PR0061    |7         |625.15       |NULL            |
|2025-11-29|PR0069    |2         |527.07       |20              |
|2025-11-26|PR0040    |16        |9850.22      |NULL            |
|2025-11-2

## 18. Analyze the campaign timeline
We summarize realized revenue and order activity by day and hour.

In [42]:
# The daily and hourly summaries provide two different views of when campaign activity took place.

# Group sales by date to understand how revenue and order volume changed during the campaign.
daily_sales_summary_df = (
    sales_df
    .groupBy("order_date")
    .agg(
        F.round(F.sum("revenue"), 2).alias("daily_revenue"),
        F.sum("quantity").alias("units_sold"),
        F.countDistinct("order_id").alias("order_count")
    )
    .orderBy("order_date")
)

# Group by hour and channel to compare when customers placed orders through each channel.
hourly_channel_sales_df = (
    sales_df
    .groupBy("order_hour", "channel")
    .agg(
        F.round(F.sum("revenue"), 2).alias("total_revenue"),
        F.countDistinct("order_id").alias("order_count"),
        F.sum("quantity").alias("units_sold")
    )
    .orderBy("order_hour", "channel")
)

In [43]:
# Show the timeline outputs separately so changes by date and by hour are easy to inspect.

# Display the daily campaign trend.
daily_sales_summary_df.show(30, truncate=False)

# Display the hourly sales split by channel.
hourly_channel_sales_df.show(50, truncate=False)

+----------+-------------+----------+-----------+
|order_date|daily_revenue|units_sold|order_count|
+----------+-------------+----------+-----------+
|2025-11-20|320780.53    |906       |245        |
|2025-11-21|249342.77    |733       |224        |
|2025-11-22|275835.16    |835       |238        |
|2025-11-23|316165.32    |922       |244        |
|2025-11-24|243902.85    |814       |241        |
|2025-11-25|274886.35    |784       |229        |
|2025-11-26|289815.59    |913       |243        |
|2025-11-27|298764.24    |861       |233        |
|2025-11-28|230809.09    |739       |226        |
|2025-11-29|195621.18    |599       |198        |
|2025-11-30|250163.81    |725       |216        |
+----------+-------------+----------+-----------+

+----------+-------+-------------+-----------+----------+
|order_hour|channel|total_revenue|order_count|units_sold|
+----------+-------+-------------+-----------+----------+
|0         |NULL   |20659.02     |17         |59        |
|0         |mobil

## 19. Apply window functions to business questions
We use ranking and cumulative calculations to understand product performance and campaign progress.

In [44]:
# Ranking within category makes the result useful even when categories have very different sales volumes.

# Window 1: Rank products by revenue within their own category.
# partitionBy restarts the ranking for each category.
category_rank_window = Window.partitionBy(
    "category"
).orderBy(
    F.desc("total_revenue")
)

ranked_products_df = product_performance_df.withColumn(
    "revenue_rank",
    F.rank().over(category_rank_window)
)

# Keep the category and rank together so the leading products are easy to compare.
ranked_products_df.orderBy(
    "category", "revenue_rank"
).show(50, truncate=False)

+----------+-------------------+-----------+-------------+-----------+----------+--------------------+------------+
|product_id|product_name       |category   |total_revenue|order_count|units_sold|average_discount_pct|revenue_rank|
+----------+-------------------+-----------+-------------+-----------+----------+--------------------+------------+
|PR0020    |Eco Tablet 20      |Beauty     |55699.55     |41         |156       |18.41               |1           |
|PR0049    |Nova Blender 49    |Beauty     |42024.99     |32         |118       |20.16               |2           |
|PR0100    |Prime Jacket 100   |Beauty     |36888.5      |32         |122       |19.06               |3           |
|PR0073    |Urban Shoes 73     |Beauty     |33442.51     |32         |110       |19.06               |4           |
|PR0092    |Prime Lamp 92      |Beauty     |33176.73     |28         |107       |14.64               |5           |
|PR0023    |Prime Blender 23   |Beauty     |29807.75     |22         |88

In [45]:
# Keeping ties at rank one is intentional, since more than one product can lead on the same date.

# Window 2: Rank products by units sold separately for each campaign date.
# This helps identify the best-selling product on every day.
daily_product_rank_window = Window.partitionBy(
    "order_date"
).orderBy(
    F.desc("units_sold")
)

daily_best_sellers_df = (
    daily_product_sales_df
    .withColumn(
        "daily_rank",
        F.rank().over(daily_product_rank_window)
    )
    .filter(F.col("daily_rank") == 1)
    .orderBy("order_date")
)

# rank() keeps multiple products when they share the highest daily unit count.
daily_best_sellers_df.show(50, truncate=False)

+----------+----------+----------+-------------+----------+
|order_date|product_id|units_sold|daily_revenue|daily_rank|
+----------+----------+----------+-------------+----------+
|2025-11-20|PR0001    |32        |11254.39     |1         |
|2025-11-21|PR0003    |39        |14641.85     |1         |
|2025-11-22|PR0095    |35        |16259.18     |1         |
|2025-11-23|PR0074    |37        |15964.55     |1         |
|2025-11-24|PR0001    |45        |13918.24     |1         |
|2025-11-25|PR0043    |38        |21540.43     |1         |
|2025-11-26|PR0020    |33        |16433.61     |1         |
|2025-11-27|PR0079    |30        |12140.4      |1         |
|2025-11-27|PR0005    |30        |13410.53     |1         |
|2025-11-28|PR0087    |29        |10406.93     |1         |
|2025-11-29|PR0092    |23        |5002.15      |1         |
|2025-11-30|PR0006    |27        |12448.18     |1         |
+----------+----------+----------+-------------+----------+



In [46]:
# The running total follows campaign dates in order, so each row includes revenue up to that point.

# Window 3: Calculate the running campaign revenue from the first date to the current date.
# rowsBetween makes the calculation include all earlier dates and the current date.
campaign_running_window = (
    Window.orderBy("order_date")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

cumulative_sales_df = daily_sales_summary_df.withColumn(
    "cumulative_revenue",
    F.round(
        F.sum("daily_revenue").over(campaign_running_window),
        2
    )
)

# The final value represents the total realized revenue across the campaign dates.
cumulative_sales_df.show(30, truncate=False)

+----------+-------------+----------+-----------+------------------+
|order_date|daily_revenue|units_sold|order_count|cumulative_revenue|
+----------+-------------+----------+-----------+------------------+
|2025-11-20|320780.53    |906       |245        |320780.53         |
|2025-11-21|249342.77    |733       |224        |570123.3          |
|2025-11-22|275835.16    |835       |238        |845958.46         |
|2025-11-23|316165.32    |922       |244        |1162123.78        |
|2025-11-24|243902.85    |814       |241        |1406026.63        |
|2025-11-25|274886.35    |784       |229        |1680912.98        |
|2025-11-26|289815.59    |913       |243        |1970728.57        |
|2025-11-27|298764.24    |861       |233        |2269492.81        |
|2025-11-28|230809.09    |739       |226        |2500301.9         |
|2025-11-29|195621.18    |599       |198        |2695923.08        |
|2025-11-30|250163.81    |725       |216        |2946086.89        |
+----------+-------------+--------

## 20. Run analytical queries using Spark SQL
We register the curated DataFrames as temporary views and answer three business questions with SQL.

In [47]:
# Temporary views let the SQL examples use the same cleaned datasets as the DataFrame work.

# Register the same curated data used in the DataFrame analysis.
# This ensures SQL follows the same quality rules and revenue definitions.
sales_enriched_df.createOrReplaceTempView("sales")
valid_inventory_df.createOrReplaceTempView("inventory")

In [48]:
# This query is the SQL version of the category performance question answered earlier.

# SQL question 1: Which product categories generated the highest realized revenue?
category_sql_df = spark.sql("""
    SELECT
        category,
        ROUND(SUM(revenue), 2) AS total_revenue,
        COUNT(DISTINCT order_id) AS order_count,
        SUM(quantity) AS units_sold
    FROM sales
    GROUP BY category
    ORDER BY total_revenue DESC
""")

category_sql_df.show(truncate=False)

+-----------+-------------+-----------+----------+
|category   |total_revenue|order_count|units_sold|
+-----------+-------------+-----------+----------+
|Electronics|708814.73    |597        |2110      |
|Toys       |555280.62    |502        |1717      |
|Sports     |519506.21    |425        |1486      |
|Fashion    |419770.68    |328        |1212      |
|Beauty     |400890.63    |376        |1265      |
|Home       |341824.02    |309        |1041      |
+-----------+-------------+-----------+----------+



In [49]:
# The channel query uses the same core measures to keep the comparison consistent.

# SQL question 2: How do the sales channels compare in revenue and order volume?
channel_sql_df = spark.sql("""
    SELECT
        channel,
        ROUND(SUM(revenue), 2) AS total_revenue,
        COUNT(DISTINCT order_id) AS order_count,
        SUM(quantity) AS units_sold
    FROM sales
    GROUP BY channel
    ORDER BY total_revenue DESC
""")

channel_sql_df.show(truncate=False)

+-------+-------------+-----------+----------+
|channel|total_revenue|order_count|units_sold|
+-------+-------------+-----------+----------+
|web    |1131708.17   |962        |3366      |
|mobile |620718.75    |531        |1890      |
|store  |618534.41    |534        |1881      |
|NULL   |575125.56    |510        |1694      |
+-------+-------------+-----------+----------+



In [50]:
# Inventory movement is grouped by both warehouse and event type to retain the operational detail.

# SQL question 3: What stock movement was recorded by each warehouse and event type?
inventory_sql_df = spark.sql("""
    SELECT
        warehouse,
        event_type,
        SUM(stock_change) AS net_stock_change,
        COUNT(*) AS event_count
    FROM inventory
    GROUP BY warehouse, event_type
    ORDER BY warehouse, event_type
""")

inventory_sql_df.show(truncate=False)

+---------+---------------+----------------+-----------+
|warehouse|event_type     |net_stock_change|event_count|
+---------+---------------+----------------+-----------+
|WH-E     |DAMAGE         |-175            |23         |
|WH-E     |RESTOCK        |-195            |41         |
|WH-E     |RETURN         |40              |20         |
|WH-E     |SALE_ADJUSTMENT|-60             |28         |
|WH-N     |DAMAGE         |-125            |23         |
|WH-N     |RESTOCK        |245             |42         |
|WH-N     |RETURN         |20              |18         |
|WH-N     |SALE_ADJUSTMENT|335             |24         |
|WH-S     |DAMAGE         |-165            |28         |
|WH-S     |RESTOCK        |-180            |56         |
|WH-S     |RETURN         |65              |13         |
|WH-S     |SALE_ADJUSTMENT|165             |26         |
|WH-W     |DAMAGE         |-175            |22         |
|WH-W     |RESTOCK        |-35             |36         |
|WH-W     |RETURN         |45  

## 21. Validate SQL results against DataFrame results
We compare equivalent aggregation totals to confirm that both Spark APIs follow the same calculations.

In [51]:
# Compare totals rather than individual rows here, because the two approaches return different result shapes.

# Calculate the total revenue from the category DataFrame result.
category_df_total = (
    category_performance_df
    .agg(F.sum("total_revenue").alias("value"))
    .first()["value"] or 0.0
)

# Calculate the same total using the SQL result.
category_sql_total = (
    category_sql_df
    .agg(F.sum("total_revenue").alias("value"))
    .first()["value"] or 0.0
)

# Compare the channel totals from both implementations.
channel_df_total = (
    channel_performance_df
    .agg(F.sum("total_revenue").alias("value"))
    .first()["value"] or 0.0
)

channel_sql_total = (
    channel_sql_df
    .agg(F.sum("total_revenue").alias("value"))
    .first()["value"] or 0.0
)

# Compare total inventory movement from the DataFrame and SQL results.
inventory_df_total = (
    valid_inventory_df
    .agg(F.sum("stock_change").alias("value"))
    .first()["value"] or 0
)

inventory_sql_total = (
    inventory_sql_df
    .agg(F.sum("net_stock_change").alias("value"))
    .first()["value"] or 0
)


In [52]:
# These assertions make the comparison repeatable and will stop the notebook if the totals drift.


# Allow a one-cent difference for rounded revenue totals.
assert abs(category_df_total - category_sql_total) < 0.01
assert abs(channel_df_total - channel_sql_total) < 0.01

# Stock changes are integers, so their totals should match exactly.
assert inventory_df_total == inventory_sql_total

print(
    "SQL and DataFrame totals match for category, channel and inventory analyses."
)

SQL and DataFrame totals match for category, channel and inventory analyses.


## 22. Inspect execution plans and optimization
We inspect Spark's execution plans to identify shuffles and demonstrate a broadcast join.

In [53]:
# The formatted plan is useful for checking where joins, exchanges and other execution steps occur.

# Plan 1: Inspect how Spark enriches sales with product and customer information.
# Look for Exchange stages, which commonly indicate data shuffling.
sales_enriched_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (84)
+- Project (83)
   +- BroadcastHashJoin LeftOuter BuildRight (82)
      :- Project (68)
      :  +- BroadcastHashJoin LeftOuter BuildRight (67)
      :     :- Project (53)
      :     :  +- Filter (52)
      :     :     +- BroadcastHashJoin LeftOuter BuildRight (51)
      :     :        :- Project (36)
      :     :        :  +- BroadcastHashJoin LeftOuter BuildRight (35)
      :     :        :     :- Project (20)
      :     :        :     :  +- BroadcastHashJoin LeftOuter BuildRight (19)
      :     :        :     :     :- HashAggregate (6)
      :     :        :     :     :  +- Exchange (5)
      :     :        :     :     :     +- HashAggregate (4)
      :     :        :     :     :        +- Project (3)
      :     :        :     :     :           +- Filter (2)
      :     :        :     :     :              +- Scan csv  (1)
      :     :        :     :     +- BroadcastExchange (18)
      :     :        :     :        +- Project (17)
    

In [54]:
# Reviewing the aggregation plan helps identify whether the grouping introduces a shuffle.

# Plan 2: Inspect the aggregation plan used to calculate category performance.
# Grouping and sorting may require Spark to move data between partitions.
category_performance_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (91)
+- Sort (90)
   +- Exchange (89)
      +- HashAggregate (88)
         +- Exchange (87)
            +- HashAggregate (86)
               +- HashAggregate (85)
                  +- Exchange (84)
                     +- HashAggregate (83)
                        +- Project (82)
                           +- BroadcastHashJoin LeftOuter BuildRight (81)
                              :- Project (68)
                              :  +- BroadcastHashJoin LeftOuter BuildRight (67)
                              :     :- Project (53)
                              :     :  +- Filter (52)
                              :     :     +- BroadcastHashJoin LeftOuter BuildRight (51)
                              :     :        :- Project (36)
                              :     :        :  +- BroadcastHashJoin LeftOuter BuildRight (35)
                              :     :        :     :- Project (20)
                              :     :        :     :  +- Broadc

In [55]:
# This example is worth checking in the physical plan; the broadcast hint should be verified, not assumed.

# Optimization example: broadcast the product lookup table when it is small enough.
# Broadcasting sends the small lookup data to executors and can avoid shuffling
# the larger sales dataset for this join.
broadcast_product_join_df = sales_df.join(
    broadcast(
        products_unique_df.select(
            "product_id",
            "product_name",
            "category",
            "brand"
        )
    ),
    "product_id",
    "left"
)

# Inspect the plan to check whether Spark selected a broadcast hash join.
broadcast_product_join_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (69)
+- Project (68)
   +- BroadcastHashJoin LeftOuter BuildRight (67)
      :- Project (53)
      :  +- Filter (52)
      :     +- BroadcastHashJoin LeftOuter BuildRight (51)
      :        :- Project (36)
      :        :  +- BroadcastHashJoin LeftOuter BuildRight (35)
      :        :     :- Project (20)
      :        :     :  +- BroadcastHashJoin LeftOuter BuildRight (19)
      :        :     :     :- HashAggregate (6)
      :        :     :     :  +- Exchange (5)
      :        :     :     :     +- HashAggregate (4)
      :        :     :     :        +- Project (3)
      :        :     :     :           +- Filter (2)
      :        :     :     :              +- Scan csv  (1)
      :        :     :     +- BroadcastExchange (18)
      :        :     :        +- Project (17)
      :        :     :           +- Filter (16)
      :        :     :              +- HashAggregate (15)
      :        :     :                 +- Exchange (14)
      :   

## 23. Analyze product revenue concentration
We calculate each product's revenue share and cumulative contribution to campaign revenue.

In [56]:
# Revenue share is calculated against the total product revenue from the same performance dataset.

# Calculate the total product revenue used as the denominator for revenue shares.
total_product_revenue = (
    product_performance_df
    .agg(F.sum("total_revenue").alias("total_revenue"))
    .first()["total_revenue"] or 0.0
)

# Define a stable ordering so products with equal revenue have a consistent position.
concentration_window = (
    Window.orderBy(
        F.desc("total_revenue"),
        F.asc("product_id")
    )
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

# Calculate each product's contribution and its running share of total revenue.
product_concentration_df = (
    product_performance_df
    .withColumn(
        "revenue_share_pct",
        F.when(
            F.lit(total_product_revenue) > 0,
            F.round(
                F.col("total_revenue")
                / F.lit(total_product_revenue) * 100,
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "cumulative_revenue",
        F.sum("total_revenue").over(concentration_window)
    )
    .withColumn(
        "cumulative_revenue_share_pct",
        F.when(
            F.lit(total_product_revenue) > 0,
            F.round(
                F.col("cumulative_revenue")
                / F.lit(total_product_revenue) * 100,
                2
            )
        ).otherwise(F.lit(0.0))
    )
)

# Show the leading products and how much of campaign revenue they contribute.
product_concentration_df.select(
    "product_id",
    "product_name",
    "category",
    "total_revenue",
    "revenue_share_pct",
    "cumulative_revenue_share_pct"
).show(15, truncate=False)

+----------+-------------------+-----------+-------------+-----------------+----------------------------+
|product_id|product_name       |category   |total_revenue|revenue_share_pct|cumulative_revenue_share_pct|
+----------+-------------------+-----------+-------------+-----------------+----------------------------+
|PR0001    |Nova Tablet 1      |Electronics|70507.18     |2.39             |2.39                        |
|PR0095    |Urban Headphones 95|Electronics|64446.77     |2.19             |4.58                        |
|PR0020    |Eco Tablet 20      |Beauty     |55699.55     |1.89             |6.47                        |
|PR0043    |Prime Headphones 43|Electronics|50010.86     |1.7              |8.17                        |
|PR0012    |Prime Tablet 12    |Toys       |49484.86     |1.68             |9.85                        |
|PR0087    |Eco Watch 87       |Sports     |46683.21     |1.58             |11.43                       |
|PR0014    |Nova Tablet 14     |Sports     |45

## 24. Create a compact data-quality summary
We bring the main source, valid and quarantined record counts into one summary.

In [57]:
# Collecting these counts in one place makes the quality decisions easier to explain to a reviewer.

# Collect the important row counts so the reviewer can understand the cleaning results.
quality_summary_data = [
    ("Customers - source", customers_df.count()),
    ("Products - source", products_df.count()),
    ("Orders - source", orders_df.count()),
    ("Orders - exact duplicate extra rows", exact_duplicate_order_rows_df.count()),
    ("Orders - valid after quality rules", valid_orders_df.count()),
    ("Orders - quarantined", quarantine_orders_df.count()),
    ("Completed sales", sales_df.count()),
    ("Inventory - source", inventory_df.count()),
    ("Inventory - valid", valid_inventory_df.count()),
    ("Inventory - quarantined", inventory_quarantine_df.count()),
    ("Products - duplicate key rows", duplicate_product_rows_df.count()),
    ("Customers - duplicate key rows", duplicate_customer_rows_df.count())
]

# Convert the collected summary into a Spark DataFrame for display and storage.
quality_summary_df = spark.createDataFrame(
    quality_summary_data,
    ["dataset_or_check", "record_count"]
)

quality_summary_df.show(truncate=False)

+-----------------------------------+------------+
|dataset_or_check                   |record_count|
+-----------------------------------+------------+
|Customers - source                 |1500        |
|Products - source                  |100         |
|Orders - source                    |6003        |
|Orders - exact duplicate extra rows|3           |
|Orders - valid after quality rules |5025        |
|Orders - quarantined               |975         |
|Completed sales                    |2537        |
|Inventory - source                 |450         |
|Inventory - valid                  |439         |
|Inventory - quarantined            |11          |
|Products - duplicate key rows      |0           |
|Customers - duplicate key rows     |0           |
+-----------------------------------+------------+



## 25. Test important business rules
We use assertions and small test datasets to check revenue calculations, discounts and join behavior.

In [58]:
# These assertions check the key assumptions that define which records count as realized sales.

# Confirm that realized sales contain only completed orders.
assert sales_df.filter(
    F.col("order_status") != "completed"
).count() == 0

# Confirm that every sale has a positive quantity and unit price.
assert sales_df.filter(
    F.col("quantity").isNull()
    | (F.col("quantity") <= 0)
    | F.col("unit_price").isNull()
    | (F.col("unit_price") <= 0)
).count() == 0

# Confirm that every sale has a discount within the accepted range.
assert sales_df.filter(
    F.col("discount_pct").isNull()
    | (F.col("discount_pct") < 0)
    | (F.col("discount_pct") > 1)
).count() == 0

# Revenue should not be negative under the selected validation rules.
assert sales_df.filter(
    F.col("revenue") < 0
).count() == 0

# No order ID should appear more than once in the valid order dataset.
assert (
    valid_orders_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
    == 0
)

# Dimension joins should not increase or decrease the number of sales records.
assert sales_enriched_df.count() == sales_df.count()

print("Core sales, duplicate-key and join assertions passed.")

Core sales, duplicate-key and join assertions passed.


In [59]:
# Small test cases make the revenue and discount rules easier to verify without relying on source data.

# Create a few known examples to verify that the revenue formula works correctly.
revenue_test_data = [
    (2, 100.0, 0.10, 180.0),
    (1, 200.0, 0.00, 200.0),
    (3, 50.0, 0.20, 120.0),
    (1, 75.0, 1.00, 0.0)
]

# Calculate revenue using the same formula applied to the real sales data.
revenue_test_df = (
    spark.createDataFrame(
        revenue_test_data,
        [
            "quantity",
            "unit_price",
            "discount_pct",
            "expected_revenue"
        ]
    )
    .withColumn(
        "calculated_revenue",
        F.round(
            F.col("quantity")
            * F.col("unit_price")
            * (1 - F.col("discount_pct")),
            2
        )
    )
)

# Every calculated result should match its manually expected value.
assert revenue_test_df.filter(
    F.abs(
        F.col("calculated_revenue") - F.col("expected_revenue")
    ) > 0.01
).count() == 0


# Test the discount boundaries independently of the source dataset.
discount_test_df = (
    spark.createDataFrame(
        [(0.0,), (0.25,), (1.0,), (1.2,), (None,)],
        ["discount_value"]
    )
    .withColumn(
        "is_valid_discount",
        F.col("discount_value").between(0.0, 1.0)
    )
)

# Zero, 25% and 100% discounts should be accepted.
assert discount_test_df.filter(
    F.col("discount_value").isin(0.0, 0.25, 1.0)
    & (~F.col("is_valid_discount"))
).count() == 0

# A discount greater than one must not be accepted.
assert discount_test_df.filter(
    (F.col("discount_value") > 1.0)
    & F.col("is_valid_discount")
).count() == 0

# A missing discount must not be treated as a valid discount.
assert discount_test_df.filter(
    F.col("discount_value").isNull()
    & F.col("is_valid_discount")
).count() == 0

print("Revenue formula and discount boundary tests passed.")

Revenue formula and discount boundary tests passed.


## 26. Save analytical and quality outputs
We save the important results as Parquet so they can be reused without repeating the full pipeline.

In [60]:
# Save reusable outputs after the analysis so downstream work does not need to rerun every transformation.

# Save the main product and category performance tables for downstream analysis.
product_performance_df.write.mode("overwrite").parquet(
    f"{OUTPUT_DIR}/product_performance"
)

category_performance_df.write.mode("overwrite").parquet(
    f"{OUTPUT_DIR}/category_performance"
)

# Save the campaign timeline and inventory analysis results.
daily_sales_summary_df.write.mode("overwrite").parquet(
    f"{OUTPUT_DIR}/daily_sales_summary"
)

daily_inventory_df.write.mode("overwrite").parquet(
    f"{OUTPUT_DIR}/daily_inventory_movements"
)

sales_inventory_comparison_df.write.mode("overwrite").parquet(
    f"{OUTPUT_DIR}/sales_inventory_comparison"
)

# Save rejected records and audit information so quality decisions are traceable.
quarantine_orders_df.write.mode("overwrite").parquet(
    f"{OUTPUT_DIR}/quarantine_orders"
)

inventory_quarantine_df.write.mode("overwrite").parquet(
    f"{OUTPUT_DIR}/quarantine_inventory"
)

exact_duplicate_order_rows_df.write.mode("overwrite").parquet(
    f"{OUTPUT_DIR}/exact_duplicate_order_rows"
)

quality_summary_df.write.mode("overwrite").parquet(
    f"{OUTPUT_DIR}/quality_summary"
)

print("Analytical and quality outputs saved under:", OUTPUT_DIR)

Analytical and quality outputs saved under: output


In [61]:
# Reading the files back checks that the saved Parquet datasets are accessible and have the expected row counts.

# Read two saved datasets to confirm that Spark can access the Parquet outputs.
saved_product_performance_df = spark.read.parquet(
    f"{OUTPUT_DIR}/product_performance"
)

saved_daily_sales_df = spark.read.parquet(
    f"{OUTPUT_DIR}/daily_sales_summary"
)

# Compare the saved row counts with the original DataFrames.
assert (
    saved_product_performance_df.count()
    == product_performance_df.count()
)

assert (
    saved_daily_sales_df.count()
    == daily_sales_summary_df.count()
)

print("Parquet read-back checks passed.")

Parquet read-back checks passed.


## 27. Stakeholder findings
Use the measured outputs above to summarize campaign performance, data quality and inventory observations.

In [62]:
# Bring the main findings together here; these are the outputs to use when writing the stakeholder summary.

# Display the overall revenue comparison for the final business summary.
revenue_comparison_df.show(truncate=False)

# Display the leading categories and products by realized revenue.
print("Top categories:")
category_performance_df.show(5, truncate=False)

print("Top products:")
product_performance_df.show(10, truncate=False)

# Display the leading channel and region results.
print("Channel performance:")
channel_performance_df.show(truncate=False)

print("Region performance:")
region_performance_df.show(truncate=False)

# Identify the highest-revenue campaign date.
print("Highest-revenue campaign date:")
daily_sales_summary_df.orderBy(
    F.desc("daily_revenue")
).show(1, truncate=False)

# Display the campaign's highest-volume daily product results.
print("Daily best sellers:")
daily_best_sellers_df.show(20, truncate=False)

# Display inventory movement and the data-quality summary.
print("Largest recorded inventory movements:")
inventory_summary_df.orderBy(
    F.desc(F.abs(F.col("net_stock_change")))
).show(10, truncate=False)

print("Data-quality summary:")
quality_summary_df.show(truncate=False)

+-------------+------------------+----------+
|naive_revenue|defensible_revenue|difference|
+-------------+------------------+----------+
|8103007.97   |2946086.89        |5156921.08|
+-------------+------------------+----------+

Top categories:
+-----------+-------------+-----------+----------+-------------------+--------------------+
|category   |total_revenue|order_count|units_sold|average_order_value|average_discount_pct|
+-----------+-------------+-----------+----------+-------------------+--------------------+
|Electronics|708814.73    |597        |2110      |1187.29            |18.79               |
|Toys       |555280.62    |502        |1717      |1106.14            |19.19               |
|Sports     |519506.21    |425        |1486      |1222.37            |19.01               |
|Fashion    |419770.68    |328        |1212      |1279.79            |18.57               |
|Beauty     |400890.63    |376        |1265      |1066.2             |18.67               |
+-----------+----

In [63]:
# This second display cell is retained as it was in the original notebook; the added note clarifies its purpose.

# Display the overall revenue comparison for the final business summary.
revenue_comparison_df.show(truncate=False)

# Display the leading categories and products by realized revenue.
print("Top categories:")
category_performance_df.show(5, truncate=False)

print("Top products:")
product_performance_df.show(10, truncate=False)

# Display the leading channel and region results.
print("Channel performance:")
channel_performance_df.show(truncate=False)

print("Region performance:")
region_performance_df.show(truncate=False)

# Identify the highest-revenue campaign date.
print("Highest-revenue campaign date:")
daily_sales_summary_df.orderBy(
    F.desc("daily_revenue")
).show(1, truncate=False)

# Display the campaign's highest-volume daily product results.
print("Daily best sellers:")
daily_best_sellers_df.show(20, truncate=False)

# Display inventory movement and the data-quality summary.
print("Largest recorded inventory movements:")
inventory_summary_df.orderBy(
    F.desc(F.abs(F.col("net_stock_change")))
).show(10, truncate=False)

print("Data-quality summary:")
quality_summary_df.show(truncate=False)

+-------------+------------------+----------+
|naive_revenue|defensible_revenue|difference|
+-------------+------------------+----------+
|8103007.97   |2946086.89        |5156921.08|
+-------------+------------------+----------+

Top categories:
+-----------+-------------+-----------+----------+-------------------+--------------------+
|category   |total_revenue|order_count|units_sold|average_order_value|average_discount_pct|
+-----------+-------------+-----------+----------+-------------------+--------------------+
|Electronics|708814.73    |597        |2110      |1187.29            |18.79               |
|Toys       |555280.62    |502        |1717      |1106.14            |19.19               |
|Sports     |519506.21    |425        |1486      |1222.37            |19.01               |
|Fashion    |419770.68    |328        |1212      |1279.79            |18.57               |
|Beauty     |400890.63    |376        |1265      |1066.2             |18.67               |
+-----------+----

## 28. Assumptions and limitations
These rules describe how incomplete source records and missing business information were handled.

- Only valid `completed` orders are included in realized sales.
- Returned orders are reported separately because refund amounts are not available.
- Discounts must be between `0` and `1`; values such as `1.2` remain flagged until clarified by the data owner.
- Missing discounts are not assumed to mean zero.
- Missing region and channel values remain visible as unknown groups.
- Duplicate dimension keys are excluded from lookup joins to prevent sales row multiplication.
- Inventory events are analyzed as recorded stock movements, not as a guaranteed reconciliation of sales.
- The dataset is synthetic learning data, so findings should not be treated as official retailer results.